# Weighted event sampling to `scipp`

McStas event files contain weighted events. This notebook focuses on converting those weighted events to a fixed number of normal, unit-weight events while exporting to the simple `scipp` representation.

## Imports

In [1]:
from pathlib import Path

import numpy as np

import mcstastox
from read_example import make_instrument

## Run a small McStas simulation

As in the other user-guide notebooks, use `read_example` to create a small instrument and generate a McStas NeXus event file. The `Read` interface takes the output folder, while McStas writes the file as `mccode.h5` inside that folder.

In [2]:
instr = make_instrument()
instr.set_parameters(wavelength=1.8, delta_wavelength=1.3)
instr.settings(ncount=1E6)
data = instr.backengine()
data_folder = Path(data[0].original_data_location)
data_folder


INFO: Using directory: "/Users/user/Projects/McStasToX/McStasToX/docs/user-guide/test_18"
INFO: Regenerating c-file: test.c
CFLAGS=
          
 The parameter format of sample is initialized 
 using a static {,,,} vector.
  -> Such static vectors support literal numbers ONLY.
  -> Any vector use of variables or defines must happen via a 
     DECLARE/INITIALIZE pointer.


-----------------------------------------------------------

Generating single GPU kernel or single CPU section layout: 

-----------------------------------------------------------

Generating GPU/CPU -DFUNNEL layout:

-----------------------------------------------------------
INFO: Recompiling: ./test.out
ld: warning: duplicate -rpath '/Users/user/Projects/McStasToX/McStasToX/.pixi/envs/docs/lib' ignored
ld: warning: duplicate -rpath '/Users/user/Projects/McStasToX/McStasToX/.pixi/envs/docs/lib' ignored
ld: warning: duplicate -rpath '/Users/user/Projects/McStasToX/McStasToX/.pixi/envs/docs/lib' ignored
INFO: ===
Op

PosixPath('/Users/user/Projects/McStasToX/McStasToX/docs/user-guide/test_18')

## Inspect the weighted `scipp_simple` export

`export_scipp_simple` returns one `scipp.DataArray`. Before sampling, its data values are the McStas event weights (`p`). Reading in chunks avoids assembling the complete event file while it is loaded.

In [3]:
chunk_size = 1_000
wavelength = mcstastox.Transfer(
    mcstas_name="L",
    coord_name="sim_wavelength",
    unit="angstrom",
)

with mcstastox.Read(data_folder) as loaded_data:
    weighted = loaded_data.export_scipp_simple(
        source_name="source",
        sample_name="sample_position",
        component_name="Square_1",
        extra_variables=wavelength,
        chunk_size=chunk_size,
    )

weighted

<scipp.DataArray>
Dimensions: Sizes[events:4179, ]
Coordinates:
* position                  vector3              [m]  (events)  [(-0.199441, 0.04, 1.71046), (-0.199441, 0.0133333, 1.71046), ..., (-0.230083, -0.02, 1.73617), (-0.255617, -0.00666667, 1.7576)]
* sample_position           vector3              [m]  ()  (0, 0, 2)
* sim_wavelength            float64             [Å]  (events)  [1.72175, 1.14978, ..., 2.03855, 1.91886]
* source_position           vector3              [m]  ()  (0, 0, 0)
* t                         float64              [s]  (events)  [0.00102527, 0.000682493, ..., 0.00120768, 0.00113886]
Data:
                            float64         [counts]  (events)  [1.89055e-12, 9.27496e-12, ..., 3.41934e-11, 1.01369e-11]

In [4]:
weighted.sizes, weighted.values[:5], weighted.coords["sim_wavelength"].values[:5]

({'events': 4179},
 array([1.89054986e-12, 9.27495561e-12, 1.25953714e-11, 5.17426011e-12,
        5.07717205e-12]),
 array([1.72174511, 1.1497757 , 1.53934999, 2.43080681, 1.69560924]))

## Configure sampling

`SamplingSettings` uses weighted reservoir sampling with replacement. The result contains exactly `n_samples` events, so an input event may appear more than once. A seed makes the result reproducible.

Here, the **input stream** is the sequence of event records read from the selected McStas component. Each record contains the weight (`p`), time (`t`), pixel ID (`id`), and any requested extra variables. With chunked loading, the stream is the same logical sequence as a full read: rows are visited in file order, and selected components are visited in component order.

By default, the reservoir is returned in its internal sampling order, so the output order is not meaningful. Set `ordered=True` to retain each sampled record's position in the input stream and sort the result back into that order. This changes only the arrangement of the sampled records, not which weighted sample is selected.

In [5]:
sampling = mcstastox.SamplingSettings(
    n_samples=1_000,
    seed=42,
    ordered=True,
)
sampling

SamplingSettings(n_samples=1000, seed=42, ordered=True)

## Export sampled normal events

Pass the settings with the `sampling=` keyword. Sampling happens while the input chunks are read, and the exported data values are all one count. Extra event coordinates are sampled with the same events and remain aligned. The output also has a scalar `effective_duration` coordinate: when `p` is a detector count rate, it is `n_samples / sum(p)` seconds, with variance propagated from `sum(p**2)`.

In [6]:
with mcstastox.Read(data_folder) as loaded_data:
    sampled = loaded_data.export_scipp_simple(
        source_name="source",
        sample_name="sample_position",
        component_name="Square_1",
        extra_variables=wavelength,
        chunk_size=chunk_size,
        sampling=sampling,
    )

sampled

<scipp.DataArray>
Dimensions: Sizes[events:1000, ]
Coordinates:
* effective_duration        float64              [s]  ()  1.98178e+10  1.43983e+17
* position                  vector3              [m]  (events)  [(-0.260724, -0.04, 1.76188), (-0.189227, 0.00666667, 1.70189), ..., (-0.230083, -0.02, 1.73617), (-0.230083, -0.02, 1.73617)]
* sample_position           vector3              [m]  ()  (0, 0, 2)
* sim_wavelength            float64             [Å]  (events)  [2.335, 1.33315, ..., 2.03855, 2.03855]
* source_position           vector3              [m]  ()  (0, 0, 0)
* t                         float64              [s]  (events)  [0.00138944, 0.000792082, ..., 0.00120768, 0.00120768]
Data:
                            float64         [counts]  (events)  [1, 1, ..., 1, 1]

In [7]:
assert sampled.sizes["events"] == sampling.n_samples
np.testing.assert_array_equal(sampled.values, np.ones(sampling.n_samples))
sampled.coords["effective_duration"]

sampled.sizes, sampled.values[:5], sampled.coords["sim_wavelength"].values[:5]

({'events': 1000},
 array([1., 1., 1., 1., 1.]),
 array([2.33500451, 1.33315449, 1.07931955, 2.3955878 , 2.3955878 ]))

The same settings can be used to reproduce the same event selection. Change `seed` to obtain another statistically weighted sample.

In [8]:
with mcstastox.Read(data_folder) as loaded_data:
    sampled_again = loaded_data.export_scipp_simple(
        source_name="source",
        sample_name="sample_position",
        component_name="Square_1",
        extra_variables=wavelength,
        chunk_size=chunk_size,
        sampling=sampling,
    )

np.testing.assert_array_equal(sampled.values, sampled_again.values)
np.testing.assert_array_equal(
    sampled.coords["t"].values, sampled_again.coords["t"].values
)
print("Sampling is reproducible with the same seed.")

Sampling is reproducible with the same seed.


In [9]:
sampled

<scipp.DataArray>
Dimensions: Sizes[events:1000, ]
Coordinates:
* effective_duration        float64              [s]  ()  1.98178e+10  1.43983e+17
* position                  vector3              [m]  (events)  [(-0.260724, -0.04, 1.76188), (-0.189227, 0.00666667, 1.70189), ..., (-0.230083, -0.02, 1.73617), (-0.230083, -0.02, 1.73617)]
* sample_position           vector3              [m]  ()  (0, 0, 2)
* sim_wavelength            float64             [Å]  (events)  [2.335, 1.33315, ..., 2.03855, 2.03855]
* source_position           vector3              [m]  ()  (0, 0, 0)
* t                         float64              [s]  (events)  [0.00138944, 0.000792082, ..., 0.00120768, 0.00120768]
Data:
                            float64         [counts]  (events)  [1, 1, ..., 1, 1]